# 04 RAG Agent End-to-End

Run a full agentic RAG workflow from one annual report: extract markdown with Azure Document Intelligence, build table-aware chunks, optionally persist chunks to Postgres/pgvector, inspect retrieval results, and invoke the LangChain RAG agent for a fundamentals question.

## Setup

This notebook stays thin. Document extraction, chunking, vector persistence, retrieval, and agent construction all come from reusable project modules.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from backend import ingest_reports
from market_analyst.config.settings import load_settings
from market_analyst.repositories.vector_db import full_text_search, hybrid_search, vector_search
from market_analyst.services.agent import build_market_analysis_agent, format_retrieval_results
from market_analyst.services.rag import discover_reports
from market_analyst.telemetry import configure_notebook_logging

settings = load_settings()
logger = configure_notebook_logging(run_name="04_rag_agent_end_to_end")
print("Vector collection:", settings.vector_collection_name)

## Run Configuration

Set the document and retrieval options here. `PERSIST_TO_VECTOR_DB = True` writes chunks to both the LangChain PGVector collection and the project `reports` table used for full-text search.

In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORT_INDEX = 0
MAX_PAGES = None
CHUNK_SIZE = 1400
CHUNK_OVERLAP = 180
PERSIST_TO_VECTOR_DB = True
RESET_VECTOR_COLLECTION = False
RETRIEVAL_LIMIT = 5

QUESTION = "What do the annual report chunks say about growth, debt, cash flow, and key business risks?"

## Choose A Document

The notebook discovers PDF files from `reports/`. Put the annual report PDF there, then choose `REPORT_INDEX` above.

In [ ]:
reports = discover_reports(REPORTS_DIR)
assert reports, f"No PDF reports found in {REPORTS_DIR}"

reports_df = pd.DataFrame(
    [
        {
            "index": index,
            "ticker": report.ticker,
            "company_name": report.company_name,
            "file": report.path.name,
            "path": str(report.path),
        }
        for index, report in enumerate(reports)
    ]
)
display(reports_df)

selected_report = reports[REPORT_INDEX]
print("Selected:", selected_report.ticker, selected_report.company_name, selected_report.path.name)

## Extract, Chunk, And Persist

This cell calls the shared backend ingestion path. It uses Azure Document Intelligence for markdown extraction, the service-layer table-aware RAG splitter for chunks, and the repository layer for vector/full-text persistence.

In [ ]:
result = ingest_reports(
    reports=[selected_report],
    max_pages=MAX_PAGES,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    persist=PERSIST_TO_VECTOR_DB,
    reset_collection=RESET_VECTOR_COLLECTION,
)

print(f"reports: {result.report_count}")
print(f"chunks: {result.chunk_count}")
print(f"vector ids: {len(result.vector_ids)}")
print(f"project report rows: {result.reports_rows}")

## Markdown Preview

Inspect the extracted markdown before retrieval. This is useful for checking whether tables and section headings came through cleanly.

In [ ]:
markdown_report = result.markdown_reports[0]
print(markdown_report.report.path.name, "pages:", markdown_report.page_count)
display(Markdown(markdown_report.markdown[:6000]))

## Chunk Inventory

The chunk table shows section metadata, table preservation flags, and a short preview for each chunk.

In [ ]:
chunk_rows = [
    {
        "chunk_id": chunk.id,
        "ticker": chunk.metadata.get("ticker"),
        "page": chunk.metadata.get("page_number"),
        "heading_path": chunk.metadata.get("heading_path"),
        "contains_table": bool(chunk.metadata.get("contains_table")),
        "table_format": chunk.metadata.get("table_format"),
        "chars": len(chunk.page_content),
        "preview": chunk.page_content[:240].replace("\n", " "),
    }
    for chunk in result.chunks
]
chunks_df = pd.DataFrame(chunk_rows)
display(chunks_df)

## Table Chunk Inspection

Table-aware chunks should preserve the full table plus nearby before/after context.

In [ ]:
table_chunks = [chunk for chunk in result.chunks if chunk.metadata.get("contains_table")]
print("table chunks:", len(table_chunks))

if table_chunks:
    table_chunk = table_chunks[0]
    display(Markdown(f"### {table_chunk.metadata.get('heading_path')}\n\n{table_chunk.page_content[:5000]}"))
else:
    print("No table chunks detected in the selected page range.")


## Retrieval Smoke Test

Run full-text, vector, and fused hybrid retrieval separately. This makes it clear what evidence the RAG agent can use.

In [ ]:
QUERY = QUESTION
TICKER_FILTER = selected_report.ticker

assert PERSIST_TO_VECTOR_DB, "Set PERSIST_TO_VECTOR_DB = True and rerun ingestion before retrieval."

full_text_results = full_text_search(settings, QUERY, ticker=TICKER_FILTER, limit=RETRIEVAL_LIMIT)
vector_results = vector_search(settings, QUERY, ticker=TICKER_FILTER, limit=RETRIEVAL_LIMIT)
hybrid_results = hybrid_search(settings, QUERY, ticker=TICKER_FILTER, limit=RETRIEVAL_LIMIT)

def results_frame(rows):
    return pd.DataFrame(
        [
            {
                "ticker": row.get("ticker"),
                "company_name": row.get("company_name"),
                "heading_path": (row.get("metadata") or {}).get("heading_path"),
                "full_text_rank": row.get("full_text_rank"),
                "vector_distance": row.get("vector_distance"),
                "rrf_score": row.get("rrf_score"),
                "content": str(row.get("content", ""))[:300].replace("\n", " "),
            }
            for row in rows
        ]
    )

print("Full-text results")
display(results_frame(full_text_results))
print("Vector results")
display(results_frame(vector_results))
print("Hybrid RRF results")
display(results_frame(hybrid_results))

## Retrieved Context Preview

This is the formatted context string the agent tool returns to the model.

In [ ]:
retrieval_context = format_retrieval_results(hybrid_results)
display(Markdown("```text\n" + retrieval_context[:5000] + "\n```"))


## Create The RAG Agent

The agent is built through `market_analyst.services.agent`. Its retrieval tool calls the same shared hybrid-search function used above.

In [ ]:
agent = build_market_analysis_agent(settings, retrieval_limit=RETRIEVAL_LIMIT)
print(type(agent))

## Ask A Fundamentals Question

The ticker is included in the prompt so the retrieval tool can stay scoped to the selected document.

In [ ]:
user_prompt = f"""
Ticker: {selected_report.ticker}
Company: {selected_report.company_name}
Question: {QUESTION}

Answer using the RAG retrieval tool first. Cite the retrieved sections by section/page labels where available.
""".strip()

agent_result = agent.invoke({"messages": [{"role": "user", "content": user_prompt}]})
messages = agent_result["messages"]
final_message = messages[-1]
final_text = getattr(final_message, "content", str(final_message))

display(Markdown(final_text))

## Agent Tool Trace

This compact trace confirms whether the model called the RAG retrieval tool before answering.

In [ ]:
for index, message in enumerate(messages, start=1):
    message_type = getattr(message, "type", type(message).__name__)
    tool_calls = getattr(message, "tool_calls", None)
    name = getattr(message, "name", None)
    print(f"{index}. {message_type}" + (f" | {name}" if name else ""))
    if tool_calls:
        print("   tool_calls:", tool_calls)
    content = getattr(message, "content", "")
    if content and message_type != "ai":
        print("   content:", str(content)[:500].replace("\n", " "))

## Validation

These assertions keep the notebook honest without making it brittle.

In [ ]:
assert result.report_count == 1
assert result.chunk_count > 0
assert all(chunk.metadata.get("source_path") for chunk in result.chunks)
assert all(chunk.metadata.get("heading_path") for chunk in result.chunks)
assert len(result.vector_ids) == result.chunk_count
assert result.reports_rows == result.chunk_count
assert hybrid_results, "Hybrid retrieval should return at least one chunk for the selected report."
assert messages, "Agent result should include messages."
assert str(final_text).strip(), "Agent final response should not be empty."

print("RAG agent end-to-end notebook validation passed.")